In [1]:
import numpy as np
import pandas as pd
from scipy.stats import chi2

# =========================================================
# MÉTRICAS Y TESTS DE BACKTESTING PARA VaR
# =========================================================
# Framework completo de evaluación para comparar los cuatro modelos:
#   - MLP_VaR      (red neuronal con pérdida pinball)
#   - GARCH_t      (GARCH(1,1) con distribución t-Student)
#   - HS           (Simulación Histórica)
#   - GARCH_normal (GARCH(1,1) con distribución Normal)
# =========================================================


# ---------------------------------------------------------
# UTILIDADES INTERNAS
# ---------------------------------------------------------

def hit_series(real_losses, var_preds):
    """Normaliza inputs y devuelve la serie de indicadores de violación I_t."""
    real_losses = np.asarray(real_losses, dtype=float).reshape(-1)
    var_preds   = np.asarray(var_preds,   dtype=float).reshape(-1)
    m = np.isfinite(real_losses) & np.isfinite(var_preds)
    return real_losses[m], var_preds[m], (real_losses[m] > var_preds[m]).astype(int)


def exceedance_stats(real_losses, var_preds, alpha=0.99):
    """Estadísticas básicas de frecuencia de violación."""
    real_losses, var_preds, I = hit_series(real_losses, var_preds)
    T  = len(I)
    x  = int(I.sum())
    p  = 1 - alpha
    rv = x / T if T > 0 else np.nan
    return {
        "T": T,
        "Expected Viol.": p * T,
        "Violations": x,
        "Violation Rate": rv,
        "VR Gap": abs(rv - p) if T > 0 else np.nan,
        "Coverage Bias": (rv - p) if T > 0 else np.nan,  # (+) infraestima, (-) conservador
    }


# ---------------------------------------------------------
# MÉTRICAS DE CALIDAD DE LA ESTIMACIÓN
# ---------------------------------------------------------

def tick_loss(real_losses, var_preds, alpha=0.99):
    """
    Tick Loss (Quantile Loss / Pinball Loss) al nivel alpha.

    QL = mean[ alpha * max(L - VaR, 0) + (1-alpha) * max(VaR - L, 0) ]

    Penaliza asimétricamente: una violación pesa alpha/(1-alpha) = 99×
    más que el conservadurismo. Métrica canónica para cuantiles. Cuanto menor, mejor.
    """
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    under = np.maximum(real_losses - var_preds, 0.0)   # violación: pérdida supera VaR
    over  = np.maximum(var_preds - real_losses, 0.0)   # conservadurismo excesivo
    return float(np.mean(alpha * under + (1 - alpha) * over))


def winkler_score(real_losses, var_preds, alpha=0.99):
    """
    Winkler Score adaptado a VaR unilateral.
    W = mean[ VaR + (2/alpha) * max(L - VaR, 0) ]

    Penaliza a la vez el nivel medio del VaR (coste de capital) y las violaciones.
    Cuanto menor, mejor. Modelos con VaR estructuralmente alto obtienen peor score.
    """
    real_losses, var_preds, _ = hit_series(real_losses, var_preds)
    exceedance = np.maximum(real_losses - var_preds, 0.0)
    return float(np.mean(var_preds + (2.0 / alpha) * exceedance))


def mean_var_level(var_preds):
    """VaR medio estimado. Mayor valor = modelo más conservador = mayor capital inmovilizado."""
    var_preds = np.asarray(var_preds, dtype=float)
    return float(np.nanmean(var_preds))


def violation_severity(real_losses, var_preds):
    """
    Severidad condicional en días de violación (Expected Shortfall empírico - VaR).
    Responde a: ¿cuánto pierde de más cuando el VaR falla?
    También cuenta rachas (clusters) de violaciones consecutivas.
    """
    real_losses, var_preds, I = hit_series(real_losses, var_preds)
    exc_mask = I == 1
    n_viol   = int(exc_mask.sum())

    if n_viol == 0:
        return {
            "Exc. Mean": np.nan, "Exc. Max": np.nan,
            "Exc. Median": np.nan, "Exc. Std": np.nan,
            "Clusters": 0,
        }

    exceedances = real_losses[exc_mask] - var_preds[exc_mask]

    # Rachas: nº de grupos de días consecutivos con violación
    clusters, in_cluster = 0, False
    for bit in I:
        if bit == 1 and not in_cluster:
            clusters += 1
            in_cluster = True
        elif bit == 0:
            in_cluster = False

    return {
        "Exc. Mean":   float(exceedances.mean()),
        "Exc. Max":    float(exceedances.max()),
        "Exc. Median": float(np.median(exceedances)),
        "Exc. Std":    float(exceedances.std()),
        "Clusters":    clusters,
    }


def p_value_mean(p_uc, p_cc, p_dq):
    """
    Media de p-valores UC, CC y DQ.
    Cuanto mayor, mayor holgura estadística respecto a H0 (modelo bien especificado).
    """
    vals = [v for v in [p_uc, p_cc, p_dq] if pd.notnull(v)]
    return float(np.mean(vals)) if vals else np.nan


# ---------------------------------------------------------
# TESTS ESTADÍSTICOS DE BACKTESTING
# ---------------------------------------------------------

def kupiec_test(real_losses, var_preds, alpha=0.99):
    """
    Test de Kupiec — Unconditional Coverage (UC).

    H0: tasa de violación empírica = (1 - alpha) teórica.
    LR_UC ~ chi²(1) bajo H0.
    PASS (p > 0.05): la frecuencia de violaciones es compatible con el nivel del VaR.
    """
    _, _, I = hit_series(real_losses, var_preds)
    T, x   = len(I), int(I.sum())
    p      = 1 - alpha

    if T == 0 or x == 0 or x == T:
        return {"LRuc": np.nan, "p_uc": np.nan}

    p_hat = x / T
    LRuc  = -2 * np.log(
        ((1 - p) ** (T - x) * p ** x) /
        ((1 - p_hat) ** (T - x) * p_hat ** x)
    )
    return {"LRuc": LRuc, "p_uc": 1 - chi2.cdf(LRuc, df=1)}


def christoffersen_cc_test(real_losses, var_preds, alpha=0.99):
    """
    Test de Christoffersen — Conditional Coverage (CC).

    Combina el test UC (Kupiec) con un test de independencia serial.
    H0: frecuencia correcta + violaciones independientes entre sí (sin clustering).
    LR_CC = LR_UC + LR_ind ~ chi²(2) bajo H0.
    PASS (p > 0.05): frecuencia correcta Y ausencia de clustering.
    """
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)

    if T < 2:
        return {"LRind": np.nan, "p_ind": np.nan, "LRcc": np.nan, "p_cc": np.nan}

    # Contamos transiciones entre estados de violación consecutivos
    n00 = n01 = n10 = n11 = 0
    for t in range(1, T):
        pair = (I[t - 1], I[t])
        if   pair == (0, 0): n00 += 1
        elif pair == (0, 1): n01 += 1
        elif pair == (1, 0): n10 += 1
        else:                n11 += 1

    if (n00 + n01) == 0 or (n10 + n11) == 0:
        LRind, p_ind = np.nan, np.nan
    else:
        pi01 = n01 / (n00 + n01)   # P(violación | no violación ayer)
        pi11 = n11 / (n10 + n11)   # P(violación | violación ayer)
        pi   = (n01 + n11) / (n00 + n01 + n10 + n11)

        L0   = ((1 - pi) ** (n00 + n10)) * (pi ** (n01 + n11))
        L1   = ((1 - pi01) ** n00) * (pi01 ** n01) * ((1 - pi11) ** n10) * (pi11 ** n11)

        if L0 <= 0 or L1 <= 0:
            LRind, p_ind = np.nan, np.nan
        else:
            LRind = -2 * np.log(L0 / L1)
            p_ind = 1 - chi2.cdf(LRind, df=1)

    kup  = kupiec_test(real_losses, var_preds, alpha=alpha)
    LRuc = kup["LRuc"]

    if np.isnan(LRuc) or (LRind is np.nan) or np.isnan(LRind):
        LRcc, p_cc = np.nan, np.nan
    else:
        LRcc = LRuc + LRind
        p_cc = 1 - chi2.cdf(LRcc, df=2)

    return {"LRind": LRind, "p_ind": p_ind, "LRcc": LRcc, "p_cc": p_cc}


def dq_test(real_losses, var_preds, alpha=0.99, lags=4):
    """
    Dynamic Quantile Test — DQ (Engle & Manganelli, 2004).

    Regresión OLS de los hit residuals (I_t - p) sobre sus propios retardos.
    H0: los residuos de violación son serialmente no correlacionados.
    DQ ~ chi²(lags+1) bajo H0.
    PASS (p > 0.05): sin evidencia de dinámica temporal mal especificada.
    """
    _, _, I = hit_series(real_losses, var_preds)
    T = len(I)
    p = 1 - alpha

    if T <= lags + 5:
        return {"DQ": np.nan, "p_dq": np.nan}

    hit = I - p    # hit residuals centrados en cero
    y   = hit[lags:]
    X   = np.column_stack([np.ones(len(y))] + [hit[lags - j:T - j] for j in range(1, lags + 1)])
    XtX = X.T @ X

    if np.linalg.matrix_rank(XtX) < XtX.shape[0]:
        return {"DQ": np.nan, "p_dq": np.nan}

    beta   = np.linalg.inv(XtX) @ X.T @ y
    resid  = y - X @ beta
    sigma2 = (resid @ resid) / len(y)

    if sigma2 <= 0:
        return {"DQ": np.nan, "p_dq": np.nan}

    DQ    = float((beta.T @ XtX @ beta) / sigma2)
    p_dq  = 1 - chi2.cdf(DQ, df=X.shape[1])

    return {"DQ": DQ, "p_dq": p_dq}


# ---------------------------------------------------------
# EVALUACIÓN Y COMPARACIÓN DE MODELOS
# ---------------------------------------------------------

def evaluate_var_model(real_losses, var_preds, alpha=0.99):
    """Ejecuta todas las métricas y tests sobre un modelo y devuelve un dict de resultados."""
    exc  = exceedance_stats(real_losses, var_preds, alpha=alpha)
    kup  = kupiec_test(real_losses, var_preds, alpha=alpha)
    cc   = christoffersen_cc_test(real_losses, var_preds, alpha=alpha)
    dq   = dq_test(real_losses, var_preds, alpha=alpha, lags=4)
    sev  = violation_severity(real_losses, var_preds)
    tl   = tick_loss(real_losses, var_preds, alpha=alpha)
    ws   = winkler_score(real_losses, var_preds, alpha=alpha)
    mvl  = mean_var_level(var_preds)
    pvm  = p_value_mean(kup["p_uc"], cc["p_cc"], dq["p_dq"])

    return {
        "T":              exc["T"],
        "Expected Viol.": exc["Expected Viol."],
        "Violations":     exc["Violations"],
        "Violation Rate": exc["Violation Rate"],
        "VR Gap":         exc["VR Gap"],
        "Coverage Bias":  exc["Coverage Bias"],
        "Tick Loss":      tl,
        "Winkler Score":  ws,
        "Mean VaR Level": mvl,
        "LRuc":   kup["LRuc"],
        "p_UC":   kup["p_uc"],
        "LRcc":   cc["LRcc"],
        "p_CC":   cc["p_cc"],
        "DQ":     dq["DQ"],
        "p_DQ":   dq["p_dq"],
        "p_Mean": pvm,
        "Exc. Mean":   sev["Exc. Mean"],
        "Exc. Max":    sev["Exc. Max"],
        "Exc. Median": sev["Exc. Median"],
        "Exc. Std":    sev["Exc. Std"],
        "Clusters":    sev["Clusters"],
    }


def compare_var_models(models: dict, alpha=0.99, sig=0.05):
    """
    Evalúa todos los modelos y construye una tabla comparativa ordenada.
    Orden: primero modelos que pasan CC y DQ, luego por menor VR Gap, luego por menor Tick Loss.
    """
    rows = []
    for model_name, (real_losses, var_preds) in models.items():
        res = evaluate_var_model(real_losses, var_preds, alpha=alpha)
        res["Model"] = model_name
        rows.append(res)

    df = pd.DataFrame(rows)

    # Resultado de cada test con umbral de significación sig=0.05
    df["UC Test"] = np.where(df["p_UC"] > sig, "PASS", "FAIL")
    df["CC Test"] = np.where(df["p_CC"] > sig, "PASS", "FAIL")
    df["DQ Test"] = np.where(df["p_DQ"] > sig, "PASS", "FAIL")

    # Criterio de validez: superar simultáneamente CC y DQ (criterio más exigente)
    df["Valid (CC&DQ)"] = np.where(
        (df["CC Test"] == "PASS") & (df["DQ Test"] == "PASS"), "YES", "NO"
    )

    df["_valid_rank"] = np.where(df["Valid (CC&DQ)"] == "YES", 0, 1)

    col_order = [
        "Model", "T", "Expected Viol.", "Violations", "Violation Rate", "VR Gap",
        "Coverage Bias",
        "Tick Loss", "Winkler Score", "Mean VaR Level",
        "LRuc", "p_UC", "UC Test",
        "LRcc", "p_CC", "CC Test",
        "DQ",   "p_DQ", "DQ Test",
        "p_Mean",
        "Valid (CC&DQ)",
        "Exc. Mean", "Exc. Max", "Exc. Median", "Exc. Std", "Clusters",
        "_valid_rank",
    ]

    df = df[col_order].sort_values(
        ["_valid_rank", "VR Gap", "Tick Loss"],
        ascending=True
    ).reset_index(drop=True)

    return df.drop(columns="_valid_rank")


# ---------------------------------------------------------
# CARGA DE PREDICCIONES
# ---------------------------------------------------------
# Cada CSV contiene: índice de fecha, VaR predicho y pérdida real del día

nn   = pd.read_csv("nn_var_predictions_25.csv",   index_col=0, parse_dates=True)
hs   = pd.read_csv("hs_var_predictions.csv",       index_col=0, parse_dates=True)
ga_t = pd.read_csv("garch_t_var_predictions.csv",  index_col=0, parse_dates=True)
ga_n = pd.read_csv("garch_n_var_predictions.csv",  index_col=0, parse_dates=True)

# Intersectamos fechas para asegurar comparabilidad exacta (mismo conjunto de días)
idx  = nn.index.intersection(hs.index).intersection(ga_t.index).intersection(ga_n.index)
nn, hs, ga_t, ga_n = nn.loc[idx], hs.loc[idx], ga_t.loc[idx], ga_n.loc[idx]

# Diccionario de modelos: nombre → (pérdidas reales, predicciones VaR)
models = {
    "MLP_VaR":      (nn["loss_real"].values,   nn["VaR_pred"].values),
    "HS":           (hs["loss_real"].values,   hs["VaR_HS"].values),
    "GARCH_t":      (ga_t["loss_real"].values, ga_t["VaR_GARCH"].values),
    "GARCH_normal": (ga_n["loss_real"].values, ga_n["VaR_GARCH_n"].values),
}


# ---------------------------------------------------------
# TABLA PRINCIPAL DE COMPARACIÓN
# ---------------------------------------------------------

df_results = compare_var_models(models, alpha=0.99, sig=0.05)

SEP = "=" * 160
sep = "-" * 160

print(f"\n{SEP}")
print("COMPARACIÓN DE MODELOS VaR (alpha = 0.99)")
print(SEP)

disp_cols = [
    "Model", "T", "Expected Viol.", "Violations", "Violation Rate",
    "VR Gap", "Coverage Bias",
    "Tick Loss", "Winkler Score", "Mean VaR Level",
    "LRuc", "p_UC", "UC Test",
    "LRcc", "p_CC", "CC Test",
    "DQ",   "p_DQ", "DQ Test",
    "p_Mean", "Valid (CC&DQ)",
]

show_df = df_results[disp_cols].copy()

fmt = {
    "Expected Viol.": lambda x: f"{x:.2f}",
    "Violation Rate": lambda x: f"{x:.4f}",
    "VR Gap":         lambda x: f"{x:.4f}",
    "Coverage Bias":  lambda x: f"{x:+.4f}",
    "Tick Loss":      lambda x: f"{x:.6f}",
    "Winkler Score":  lambda x: f"{x:.6f}",
    "Mean VaR Level": lambda x: f"{x:.4f}",
    "LRuc":           lambda x: f"{x:.3f}" if pd.notnull(x) else "nan",
    "p_UC":           lambda x: f"{x:.3f}" if pd.notnull(x) else "nan",
    "LRcc":           lambda x: f"{x:.3f}" if pd.notnull(x) else "nan",
    "p_CC":           lambda x: f"{x:.3f}" if pd.notnull(x) else "nan",
    "DQ":             lambda x: f"{x:.3f}" if pd.notnull(x) else "nan",
    "p_DQ":           lambda x: f"{x:.3f}" if pd.notnull(x) else "nan",
    "p_Mean":         lambda x: f"{x:.3f}" if pd.notnull(x) else "nan",
}

print(show_df.to_string(index=False, formatters=fmt))

print(f"\n{sep}")
print("LEYENDA DE MÉTRICAS")
print(sep)
leyenda = [
    ("Violation Rate",  "Tasa de violación observada; ideal = 0.0100."),
    ("VR Gap",          "Distancia absoluta al 1% teórico. Cuanto menor, mejor calibración en frecuencia."),
    ("Coverage Bias",   "Sesgo con signo: (+) infravalorar riesgo, (-) modelo conservador."),
    ("Tick Loss",       "Quantile Loss al nivel alpha. Métrica canónica de cuantiles. Cuanto menor, mejor."),
    ("Winkler Score",   "Coste base (VaR diario) + penalización por violaciones. Penaliza modelos holgados."),
    ("Mean VaR Level",  "Nivel medio del VaR estimado. Indica el grado de conservadurismo del modelo."),
    ("UC Test",         "Test de Kupiec: cobertura incondicional. PASS = frecuencia compatible con 1%."),
    ("CC Test",         "Test de Christoffersen: cobertura condicional. PASS = frecuencia + independencia."),
    ("DQ Test",         "Test dinámico. PASS = sin evidencia de mala especificación dinámica."),
    ("p_Mean",          "Media de p-valores UC, CC y DQ. Mayor valor indica mayor holgura estadística."),
    ("Valid (CC&DQ)",   "YES si el modelo pasa simultáneamente CC y DQ."),
]
for k, v in leyenda:
    print(f"  {k:<18} : {v}")


# ---------------------------------------------------------
# INTERPRETACIÓN INDIVIDUAL POR MODELO
# ---------------------------------------------------------

print(f"\n{SEP}")
print("INTERPRETACIÓN POR MODELO")
print(SEP)

for _, row in df_results.iterrows():
    vr  = row["Violation Rate"]
    gap = row["VR Gap"]
    cb  = row["Coverage Bias"]

    if pd.isna(vr):
        vr_comment = "sin tasa de violación calculable"
    elif cb > 0.001:
        vr_comment = "tiende a infravalorar el riesgo (VaR demasiado ajustado)"
    elif cb < -0.001:
        vr_comment = "tiende a sobreestimar el riesgo (VaR conservador)"
    else:
        vr_comment = "calibración casi perfecta en frecuencia"

    print(f"\n{'─' * 60}")
    print(f"  {row['Model']}")
    print(f"{'─' * 60}")
    print(f"  Violations / Expected  : {int(row['Violations'])} / {row['Expected Viol.']:.2f}")
    print(f"  Violation Rate         : {vr:.4f}  →  {vr_comment}")
    print(f"  Coverage Bias          : {cb:+.4f}")
    print(f"  VR Gap                 : {gap:.4f}")
    print(f"  Tick Loss              : {row['Tick Loss']:.6f}")
    print(f"  Winkler Score          : {row['Winkler Score']:.6f}")
    print(f"  Mean VaR Level         : {row['Mean VaR Level']:.4f}")
    print(f"  Kupiec UC              : p = {row['p_UC']:.3f}  →  {row['UC Test']}")
    print(f"  Christoffersen CC      : p = {row['p_CC']:.3f}  →  {row['CC Test']}")
    print(f"  DQ Test                : p = {row['p_DQ']:.3f}  →  {row['DQ Test']}")
    print(f"  p_Mean (holgura stat.) : {row['p_Mean']:.3f}")
    print(f"  Válido (CC & DQ)       : {row['Valid (CC&DQ)']}")
    print(f"\n  — Severidad condicional (solo días de violación) —")
    print(f"  Exceso medio           : {row['Exc. Mean']:.4f}")
    print(f"  Exceso máximo          : {row['Exc. Max']:.4f}")
    print(f"  Exceso mediano         : {row['Exc. Median']:.4f}")
    print(f"  Desv. típica exceso    : {row['Exc. Std']:.4f}")
    print(f"  Rachas de violaciones  : {int(row['Clusters'])}")


# ---------------------------------------------------------
# CRITERIO DE ELECCIÓN Y COMPARACIÓN DETALLADA
# ---------------------------------------------------------

valid_df = df_results[df_results["Valid (CC&DQ)"] == "YES"].copy()

print(f"\n{SEP}")
print("CRITERIO DE ELECCIÓN DEL MEJOR MODELO")
print(SEP)

if len(valid_df) > 0:
    winner = valid_df.sort_values(["VR Gap", "Tick Loss"], ascending=True).iloc[0]["Model"]
    print("  Prioridad 1 : modelos que pasan simultáneamente CC y DQ.")
    print("  Prioridad 2 : menor VR Gap (mejor calibración en frecuencia).")
    print("  Prioridad 3 : menor Tick Loss (mejor calibración en cuantil).")
    print(f"\n  MEJOR MODELO (entre los válidos): {winner}")
else:
    winner = df_results.sort_values(["VR Gap", "Tick Loss"], ascending=True).iloc[0]["Model"]
    print("  Ningún modelo pasa simultáneamente CC y DQ.")
    print(f"  MEJOR MODELO RELATIVO: {winner}")

if len(valid_df) >= 2:
    names   = list(valid_df.sort_values(["VR Gap", "Tick Loss"]).reset_index(drop=True)["Model"])
    m_rows  = {row["Model"]: row for _, row in valid_df.iterrows()}
    best    = m_rows[names[0]]
    second  = m_rows[names[1]]

    print(f"\n{SEP}")
    print("COMPARACIÓN DETALLADA — MODELOS VÁLIDOS (CC & DQ)")
    print(SEP)

    metrics_cmp = [
        ("Violations",    lambda v: f"{int(v)}"),
        ("Expected Viol.",lambda v: f"{v:.2f}"),
        ("Violation Rate",lambda v: f"{v:.4f}"),
        ("VR Gap",        lambda v: f"{v:.4f}"),
        ("Coverage Bias", lambda v: f"{v:+.4f}"),
        ("Tick Loss",     lambda v: f"{v:.6f}"),
        ("Winkler Score", lambda v: f"{v:.6f}"),
        ("Mean VaR Level",lambda v: f"{v:.4f}"),
        ("p_UC",          lambda v: f"{v:.3f}"),
        ("p_CC",          lambda v: f"{v:.3f}"),
        ("p_DQ",          lambda v: f"{v:.3f}"),
        ("p_Mean",        lambda v: f"{v:.3f}"),
        ("Exc. Mean",     lambda v: f"{v:.4f}"),
        ("Exc. Max",      lambda v: f"{v:.4f}"),
        ("Clusters",      lambda v: f"{int(v)}"),
    ]

    print(f"\n  {'Métrica':<22}" + "".join(f"  {n:>14}" for n in names))
    print(f"  {'─' * 22}" + "".join(f"  {'─' * 14}" for _ in names))

    for metric, fmtfn in metrics_cmp:
        vals = [fmtfn(m_rows[n][metric]) for n in names]
        print(f"  {metric:<22}" + "".join(f"  {v:>14}" for v in vals))

    # Diferencias relativas en cuatro dimensiones independientes:
    #   VR Gap        → calibración en frecuencia     (menor = mejor)
    #   Coverage Bias → sesgo sistemático con signo   (más cercano a 0 = mejor)
    #   p_Mean        → holgura estadística conjunta  (mayor = mejor)
    #   Winkler Score → coste de capital              (menor = mejor; posible derrota del MLP)
    print(f"\n{sep}")
    print("  DIFERENCIAS RELATIVAS (dimensiones independientes)")
    print(sep)

    b_vr, s_vr = best["VR Gap"], second["VR Gap"]
    if pd.notnull(b_vr) and pd.notnull(s_vr) and s_vr != 0:
        diff_pct = (s_vr - b_vr) / abs(s_vr) * 100
        sign = "mejor" if diff_pct > 0 else "peor"
        print(f"  [CALIBRACIÓN FRECUENCIA]  {names[0]} tiene un VR Gap "
              f"{abs(diff_pct):.1f}% {sign} que {names[1]}"
              f"  ({b_vr:.4f} vs {s_vr:.4f})  →  ventaja MLP")

    b_cb, s_cb = best["Coverage Bias"], second["Coverage Bias"]
    if pd.notnull(b_cb) and pd.notnull(s_cb):
        b_abs, s_abs = abs(b_cb), abs(s_cb)
        if s_abs != 0:
            diff_pct = (s_abs - b_abs) / s_abs * 100
            print(f"  [SESGO SISTEMÁTICO]       {names[0]} tiene un sesgo absoluto "
                  f"{abs(diff_pct):.1f}% {'menor' if diff_pct > 0 else 'mayor'} que {names[1]}"
                  f"  ({b_cb:+.4f} vs {s_cb:+.4f})  →  {'ventaja MLP' if diff_pct > 0 else 'ventaja ' + names[1]}")

    b_pm, s_pm = best["p_Mean"], second["p_Mean"]
    if pd.notnull(b_pm) and pd.notnull(s_pm) and s_pm != 0:
        diff_pct = (b_pm - s_pm) / s_pm * 100
        sign = "mayor" if diff_pct > 0 else "menor"
        print(f"  [HOLGURA ESTADÍSTICA]     {names[0]} tiene una holgura estadística "
              f"{abs(diff_pct):.1f}% {sign} que {names[1]}"
              f"  (p_Mean {b_pm:.3f} vs {s_pm:.3f})  →  {'ventaja MLP' if diff_pct > 0 else 'ventaja ' + names[1]}")

    b_ws, s_ws = best["Winkler Score"], second["Winkler Score"]
    if pd.notnull(b_ws) and pd.notnull(s_ws) and s_ws != 0:
        diff_pct = (b_ws - s_ws) / s_ws * 100
        sign = "peor" if diff_pct > 0 else "mejor"
        print(f"  [COSTE DE CAPITAL]        {names[0]} tiene un Winkler Score "
              f"{abs(diff_pct):.1f}% {sign} que {names[1]}"
              f"  ({b_ws:.6f} vs {s_ws:.6f})  →  {'desventaja MLP (VaR más conservador)' if diff_pct > 0 else 'ventaja MLP'}")

    # ---------------------------------------------------------
    # VEREDICTO FINAL
    # ---------------------------------------------------------
    print(f"\n{SEP}")
    print("  VEREDICTO FINAL")
    print(SEP)

    vr_gap_win  = best["VR Gap"]        < second["VR Gap"]
    bias_win    = abs(best["Coverage Bias"]) < abs(second["Coverage Bias"])
    pvalue_win  = best["p_Mean"]        > second["p_Mean"]
    winkler_win = best["Winkler Score"] < second["Winkler Score"]

    print(f"\n  Ganador: {names[0]}")
    print(f"\n  Ventajas sobre {names[1]}:")

    if vr_gap_win:
        pct = (second["VR Gap"] - best["VR Gap"]) / second["VR Gap"] * 100
        print(f"    ✓  Calibración en frecuencia superior: VR Gap {pct:.1f}% menor"
              f" ({best['VR Gap']:.4f} vs {second['VR Gap']:.4f}).")
        print(f"       La tasa de violación del MLP ({best['Violation Rate']:.4f}) es casi idéntica")
        print(f"       al nivel teórico del 1%, frente al sesgo conservador del GARCH-t ({second['Violation Rate']:.4f}).")

    if bias_win:
        pct = (abs(second["Coverage Bias"]) - abs(best["Coverage Bias"])) / abs(second["Coverage Bias"]) * 100
        print(f"    ✓  Ausencia de sesgo sistemático: Coverage Bias {pct:.1f}% más cercano a cero")
        print(f"       ({best['Coverage Bias']:+.4f} vs {second['Coverage Bias']:+.4f}).")
        print(f"       El GARCH-t subestima sistemáticamente el número de violaciones,")
        print(f"       lo que indica que su VaR está sobreestimado de forma estructural.")

    if pvalue_win:
        pct = (best["p_Mean"] - second["p_Mean"]) / second["p_Mean"] * 100
        print(f"    ✓  Mayor holgura estadística en los tres tests de backtesting:")
        print(f"       p_Mean {pct:.1f}% superior ({best['p_Mean']:.3f} vs {second['p_Mean']:.3f}).")
        print(f"       El MLP no solo supera UC, CC y DQ, sino que lo hace con un margen")
        print(f"       sustancialmente mayor, lo que indica un modelo mejor especificado.")

    if winkler_win:
        print(f"    ✓  Menor Winkler Score ({best['Winkler Score']:.6f} vs {second['Winkler Score']:.6f}).")

    print(f"\n  Limitación reconocida de {names[0]}:")
    if not winkler_win:
        ws_diff = (best["Winkler Score"] - second["Winkler Score"]) / second["Winkler Score"] * 100
        print(f"    ✗  Winkler Score {ws_diff:.1f}% superior ({best['Winkler Score']:.6f} vs {second['Winkler Score']:.6f}).")
        print(f"       El VaR estimado por el MLP es más elevado en promedio (Mean VaR Level:")
        print(f"       {best['Mean VaR Level']:.4f} vs {second['Mean VaR Level']:.4f}), lo que implica un mayor")
        print(f"       coste de oportunidad en términos de capital inmovilizado. Sin embargo,")
        print(f"       este comportamiento conservador es consistente con las exigencias de")
        print(f"       cobertura de Basilea III, que penaliza la infraestimación del riesgo")
        print(f"       más severamente que la sobreestimación.")

    print(f"\n  Conclusión: ambos modelos superan los tests UC, CC y DQ. {names[0]} es el modelo")
    print(f"  preferido por su calibración prácticamente perfecta en frecuencia, la ausencia")
    print(f"  de sesgo sistemático y la mayor holgura estadística en los tres tests. Su única")
    print(f"  desventaja —un nivel de VaR medio más elevado— es una característica conservadora")
    print(f"  deseable desde el punto de vista de la gestión prudente del riesgo.")
    print(SEP)


COMPARACIÓN DE MODELOS VaR (alpha = 0.99)
       Model    T Expected Viol.  Violations Violation Rate VR Gap Coverage Bias Tick Loss Winkler Score Mean VaR Level   LRuc  p_UC UC Test   LRcc  p_CC CC Test      DQ  p_DQ DQ Test p_Mean Valid (CC&DQ)
     MLP_VaR 2506          25.06          26         0.0104 0.0004       +0.0004  0.001385      0.117108         0.1167  0.035 0.851    PASS  0.581 0.748    PASS   1.177 0.947    PASS  0.849           YES
     GARCH_t 2506          25.06          18         0.0072 0.0028       -0.0028  0.000237      0.019769         0.0197  2.228 0.136    PASS  2.488 0.288    PASS   3.296 0.655    PASS  0.359           YES
          HS 2506          25.06          31         0.0124 0.0024       +0.0024  0.000296      0.017986         0.0178  1.323 0.250    PASS  8.893 0.012    FAIL 129.160 0.000    FAIL  0.087            NO
GARCH_normal 2506          25.06          51         0.0204 0.0104       +0.0104  0.000241      0.014815         0.0146 20.869 0.000    F

In [2]:
import numpy as np

for name, (rl, vp) in models.items():
    print(f"\n{name}")
    print(f"  VaR_pred  — mean: {np.mean(vp):.4f}  median: {np.median(vp):.4f}  min: {np.min(vp):.4f}  max: {np.max(vp):.4f}")
    print(f"  loss_real — mean: {np.mean(rl):.4f}  median: {np.median(rl):.4f}  min: {np.min(rl):.4f}  max: {np.max(rl):.4f}")


MLP_VaR
  VaR_pred  — mean: 0.1167  median: 0.0989  min: -0.0718  max: 1.3912
  loss_real — mean: -0.0002  median: -0.0004  min: -0.0736  max: 0.0784

HS
  VaR_pred  — mean: 0.0178  median: 0.0158  min: 0.0138  max: 0.0238
  loss_real — mean: -0.0002  median: -0.0004  min: -0.0736  max: 0.0784

GARCH_t
  VaR_pred  — mean: 0.0197  median: 0.0172  min: 0.0088  max: 0.1135
  loss_real — mean: -0.0002  median: -0.0004  min: -0.0736  max: 0.0784

GARCH_normal
  VaR_pred  — mean: 0.0146  median: 0.0129  min: 0.0080  max: 0.0846
  loss_real — mean: -0.0002  median: -0.0004  min: -0.0736  max: 0.0784
